In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam


In [3]:
BASE_DIR  = "smartvision_dataset"

TRAIN_DIR = f'{BASE_DIR}/classification/train'
VAL_DIR   = f'{BASE_DIR}/classification/val'
TEST_DIR  = f'{BASE_DIR}/classification/test'
os.makedirs('models', exist_ok=True)

print('✅ Setup complete.')

✅ Setup complete.


In [4]:
for class_name in os.listdir(TRAIN_DIR):
    class_folder = os.path.join(TRAIN_DIR, class_name)
    print(f"{class_name}: {len(os.listdir(class_folder))} images")

airplane: 70 images
bed: 70 images
bench: 70 images
bicycle: 70 images
bird: 70 images
bottle: 70 images
bowl: 70 images
bus: 70 images
cake: 70 images
car: 70 images
cat: 70 images
chair: 70 images
couch: 70 images
cow: 70 images
cup: 70 images
dog: 70 images
elephant: 70 images
horse: 70 images
motorcycle: 70 images
person: 70 images
pizza: 70 images
potted plant: 70 images
stop sign: 70 images
traffic light: 70 images
train: 70 images
truck: 70 images


In [6]:
# IMAGE SETTINGS

IMG_SIZE  = (224, 224)
BATCH     = 32
EPOCHS    = 15
NUM_CLASSES = 26

print("ENTERED INTO DATA AUGMENTATION")

train_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.15,

    horizontal_flip=True
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("COMPLETED DATA AUGMENTATION")
print("="*32)

print("ENTER INTO DATA GENERATORS")

train_gen = train_datagen.flow_from_directory(

    TRAIN_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(

    VAL_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(

    TEST_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=False
)


print("COMPLETED DATA GENERATORS")
print("="*32)

CLASS_NAMES = list(train_gen.class_indices.keys())

print(f"\nNUMBER OF CLASSES: {len(CLASS_NAMES)}")

print(CLASS_NAMES)

ENTERED INTO DATA AUGMENTATION
COMPLETED DATA AUGMENTATION
ENTER INTO DATA GENERATORS
Found 1820 images belonging to 26 classes.
Found 390 images belonging to 26 classes.
Found 390 images belonging to 26 classes.
COMPLETED DATA GENERATORS

NUMBER OF CLASSES: 26
['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [7]:
# Load pretrained VGG16 base model
base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

print("PRETRAINED VGG16 LOAD")

PRETRAINED VGG16 LOAD


In [8]:
# Freeze all layers
for layer in base.layers:
    layer.trainable = False


In [9]:
# Custom classification head
x = base.output

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)

x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

output = layers.Dense(NUM_CLASSES, activation='softmax')(x)


In [10]:
# Final model
model = models.Model(
    inputs=base.input,
    outputs=output,
    name='VGG16'
)

In [11]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMODEL COMPILED")


MODEL COMPILED


In [12]:
# CALLBACKS
# =========================================================

callbacks_list = [

    callbacks.ModelCheckpoint(

        'models/VGG16_best.keras',

        save_best_only=True,

        monitor='val_accuracy',

        verbose=1
    ),

    callbacks.EarlyStopping(

        monitor='val_accuracy',

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    callbacks.ReduceLROnPlateau(

        monitor='val_loss',

        factor=0.5,

        patience=2,

        min_lr=1e-7,

        verbose=1
    )
]


In [13]:
# MODEL SUMMARY
# =========================================================

model.summary()

Model: "VGG16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,32

 Total params: 15,117,402 (57.67 MB)

 Trainable params: 401,690 (1.53 MB)

 Non-trainable params: 14,715,712 (56.14 MB)

In [ ]:
# INITIAL TRAINING
# =========================================================

print("\nSTARTING INITIAL TRAINING")

history = model.fit(

    train_gen,

    validation_data=val_gen,

    epochs=EPOCHS,

    callbacks=callbacks_list
)



STARTING INITIAL TRAINING
Epoch 1/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.0396 - loss: 3.8104
Epoch 1: val_accuracy improved from None to 0.04872, saving model to models/VGG16_best.keras

Epoch 1: finished saving model to models/VGG16_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 243s 4s/step - accuracy: 0.0473 - loss: 3.6391 - val_accuracy: 0.0487 - val_loss: 3.2634 - learning_rate: 1.0000e-04
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1061 - loss: 3.2564
Epoch 2: val_accuracy did not improve from 0.04872
57/57 ━━━━━━━━━━━━━━━━━━━━ 236s 4s/step - accuracy: 0.1209 - loss: 3.1628 - val_accuracy: 0.0410 - val_loss: 3.2739 - learning_rate: 1.0000e-04
Epoch 3/15
34/57 ━━━━━━━━━━━━━━━━━━━━ 1:19 3s/step - accuracy: 0.1950 - loss: 2.8701